Ссылка на урок: [Спринт 2/5: 2 RAG: от сырых данных до точных ответов → Тема 4/4: Оценка качества RAG → Урок 3/4: Метрики генерации и LLM-as-judge]( https://practicum.yandex.ru/learn/ai-engineer/courses/99ad0dee-d9e6-4da1-b921-64e7e9653a4e/sprints/1104074/topics/68295d2a-b17c-4564-8957-808c64a8e401/lessons/5f3fe08b-ddd4-4327-8cc1-a75464fb765c/#76ffa4f4-731a-4575-950e-47b3d4eb15c1 )

In [13]:
import json

import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer

with open("corpus.json", encoding="utf-8") as f:
    data = json.load(f)
chunks = data["chunks"]
queries = data["queries"]
judge_cases = data["judge_cases"]
texts = [c["text"] for c in chunks]

print(
    "Чанков:", len(chunks),
    "| Запросов:", len(queries),
    "| Кейсов:", len(judge_cases),
)
# Чанков: 41 | Запросов: 20 | Кейсов: 10

# Retrieval: тот же multilingual-e5-base с префиксами, что во всей теме
embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="cuda")
doc_vecs = embedder.encode(
    ["passage: " + t for t in texts],
    normalize_embeddings=True,
    show_progress_bar=False,
)


def dense_search(query, k=3):
    """Возвращает индексы top-k чанков по косинусному сходству."""
    q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
    return [int(i) for i in np.argsort(q_vec @ doc_vecs.T)[::-1][:k]]


def build_context(chunk_ids):
    """Склеивает чанки в нумерованный контекст."""
    return "\n".join(f"[{n}] {texts[i]}" for n, i in enumerate(chunk_ids, 1))


# Языковая модель: тот же локальный сервер, что и в прошлых уроках
client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
LLM = "qwen3.5"
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

(5, 'Утверждения в ответе точно соответствуют фактам, изложенным во фрагментах. Указанное время гарантии (три года), перечень исключений (залитие, удары, вскрытие корпуса, адаптеры стороннего производителя) и условия возврата товара (два недели,')
(5, 'Ответ полностью соответствует вопросу: он прямо указывает на срок гарантии (три года) и перечисляет конкретные исключения (залитие, удары, вскрытие, сторонние адаптеры), закрывая обе части запроса.')


In [14]:
# Используем client, LLM и NO_THINK из общего кода

JUDGE_SYSTEM = (
    "Ты — строгий эксперт по оценке справочных ответов. Оценивай "
    "беспристрастно, длина ответа не является достоинством. "
    "Отвечай только JSON-объектом."
)

FAITH_PROMPT = """Оцени ВЕРНОСТЬ КОНТЕКСТУ (faithfulness):
следует ли каждое утверждение ответа из фрагментов.
Соответствие вопросу не оценивай — только происхождение фактов.

Шкала:
5 — все утверждения следуют из фрагментов;
4 — добавлено только безобидное обобщение;
3 — добавлена неподтверждённая деталь;
2 — часть утверждений выдумана или противоречит фрагментам;
1 — ключевые факты отсутствуют во фрагментах или искажены.
Ответ «В базе знаний ответа нет» ничего не утверждает и получает 5.

Фрагменты:
{context}

Ответ: {answer}

Верни JSON с полями reason и score."""

REL_PROMPT = """Оцени РЕЛЕВАНТНОСТЬ (answer relevance):
отвечает ли ответ на заданный вопрос по существу.
Правдивость не проверяй — только соответствие вопросу.

Шкала:
5 — прямой и полный ответ;
4 — ответ по существу с небольшими отступлениями;
3 — закрыта только часть вопроса;
2 — текст на нужную тему, но ответа нет;
1 — другая тема или отказ отвечать.

Вопрос: {question}

Ответ: {answer}

Верни JSON с полями reason и score."""

JUDGE_RESPONSE_FORMAT = {
    "type": "json_object",
    "schema": {
        "type": "object",
        "properties": {
            "reason": {"type": "string", "maxLength": 240},
            "score": {
                "type": "integer",
                "minimum": 1,
                "maximum": 5,
            },
        },
        "required": ["reason", "score"],
        "additionalProperties": False,
    },
}


def judge(prompt):
    """Возвращает оценку 1–5 и обоснование LLM-судьи."""
    # Реализуйте вызов модели, разбор JSON и проверку оценки
    response = client.chat.completions.create(
        model=LLM,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=200,
        response_format=JUDGE_RESPONSE_FORMAT,
        extra_body=NO_THINK,
    )
    value = json.loads(response.choices[0].message.content)
    score = value["score"]
    reason = value["reason"]

    if not isinstance(score, int) or not 1 <= score <= 5:
        raise ValueError(f"Некорректная оценка судьи: {score!r}")
    if not isinstance(reason, str) or not reason.strip():
        raise ValueError("Судья не вернул обоснование")

    return score, reason

def judge_faithfulness(context, answer):
    """Оценивает верность ответа контексту."""
    prompt = FAITH_PROMPT.format(context=context, answer=answer)
    return judge(prompt)


def judge_relevance(question, answer):
    """Оценивает соответствие ответа вопросу."""
    prompt = REL_PROMPT.format(question=question, answer=answer)
    return judge(prompt)


# Проверка на первом размеченном кейсе: обе функции возвращают
# кортеж «(оценка, обоснование)»
case = judge_cases[0]
context = build_context(case["context_ids"])
print(judge_faithfulness(context, case["answer"]))
print(judge_relevance(case["query"], case["answer"]))
# (5, '<обоснование судьи>')
# (5, '<обоснование судьи>')

Средний faithfulness: 3.7
Средний answer relevance: 4.1
Точное совпадение faith: 5
Точное совпадение relevance: 8
Порог ниже 3 | faith 7 | relevance 9
Порог ниже 4 | faith 10 | relevance 9
Проблемные ответы по рабочему порогу:
{'id': 2, 'faith': 2, 'rel': 5, 'faith_reason': 'Утверждение о том, что гарантия действует семь лет, противоречит фрагменту [1], где указано, что бесплатный ремонт предоставляется в течение трёх лет. Также в ответе отсутствует упоминание об адаптерах стороннего производителя, что указано,', 'rel_reason': 'Ответ содержит прямую информацию о сроке гарантии (7 лет) и перечисляет конкретные исключения (залитие, удары, вскрытие), полностью закрывая оба аспекта вопроса.'}
{'id': 3, 'faith': 2, 'rel': 5, 'faith_reason': 'Утверждение в ответе содержит фактические ошибки, не соответствующие фрагментам. Фрагмент [1] указывает, что у коммутатора Орбита-12 24 гнезда, тогда как ответ утверждает наличие 48 портов. Кроме того, фрагмент [2] связывает наличие двух бл', 'rel_reaso

In [16]:
# Используем judge_cases, build_context() и функции судьи из предыдущего кода

# Четыре параллельных списка: по одному значению на каждый из 10 кейсов.
# judge_* — оценки нашего судьи, human_* — оценки асессора из corpus.json
judge_faith, judge_rel = [], []
human_faith, human_rel = [], []
# judged_rows — оценки обоих судей вместе с обоснованиями,
# чтобы потом напечатать проблемные ответы
judged_rows = []

for case in judge_cases:
    context = build_context(case["context_ids"])
    faith, faith_reason = judge_faithfulness(context, case["answer"])
    rel, rel_reason = judge_relevance(case["query"], case["answer"])
    judge_faith.append(faith)
    judge_rel.append(rel)
    human_faith.append(case["human_faithfulness"])
    human_rel.append(case["human_relevance"])
    judged_rows.append(
        {
            "id": case["id"],
            "faith": faith,
            "rel": rel,
            "faith_reason": faith_reason,
            "rel_reason": rel_reason,
        }
    )


def decision_agreement(scores, human_scores, threshold):
    """Считает совпадения решений по заданному порогу."""
    return sum(
        (score < threshold) == (human_score < threshold)
        for human_score, score in zip(human_scores, scores)
    )


print("Средний faithfulness:", round(float(np.mean(judge_faith)), 1))
print("Средний answer relevance:", round(float(np.mean(judge_rel)), 1))
print(
    "Точное совпадение faith:",
    sum(a == b for a, b in zip(judge_faith, human_faith)),
)
print(
    "Точное совпадение relevance:",
    sum(a == b for a, b in zip(judge_rel, human_rel)),
)
for threshold in (3, 4):
    print(
        "Порог ниже", threshold,
        "| faith", decision_agreement(judge_faith, human_faith, threshold),
        "| relevance", decision_agreement(judge_rel, human_rel, threshold),
    )

print("Проблемные ответы по рабочему порогу:")
for row in judged_rows:
    if row["faith"] < 4 or row["rel"] < 3:
        print(row)


Вопрос: два устройства получили одинаковый IP-адрес, что делать
Ответ: Здравствуйте! Ситуация, когда два устройства получают одинаковый IP-адрес, известна как **конфликт адресов**, и она может привести к нестабильной работе сети, потерям пакетов данных и сбоям в подключении.

На основе предоставленных данных и общих принципов работы сетевого оборудования, вот пошаговый план действий для решения этой проблемы:

1.  **Идентификация конфликта**
    Код ошибки **E-41** прямо указывает на то, что произошло именно это: два устройства одновременно заняли один и тот же IP-адрес. Это может произойти, если DHCP-сервер (обычно встроенный в роутер) случайно выдал адрес двум разным клиентам, либо если одно из устройств имеет статический IP, который пересекается с диапазоном динамических адресов.

2.  **Решение: Настройка статических адресов**
    Самый надежный способ устранить конфликт — исключить автоматическое назначение адреса для одного из устройств.
    *   Зайдите в настройки сетевого обору

In [ ]:
HANDED_OVER = (
    "Ты — дружелюбный ассистент поддержки. Используй фрагменты базы "
    "знаний и свои знания о сетевом оборудовании. Отвечай развёрнуто."
)
REFUSAL = "В базе знаний ответа нет"


def rag_answer(query, system, k=3):
    """Возвращает RAG-ответ и индексы найденных чанков."""
    ids = dense_search(query, k)
    user = (
        f"Фрагменты базы знаний:\n{build_context(ids)}"
        f"\n\nВопрос: {query}"
    )
    response = client.chat.completions.create(
        model=LLM,
        messages=[
            {"role": "system", "content": system},
            {"role": "user", "content": user},
        ],
        temperature=0.0,
        max_tokens=2000,
        extra_body=NO_THINK,
    )
    return response.choices[0].message.content.strip(), ids


def is_refusal(answer):
    """Проверяет наличие договорной фразы отказа."""
    normalized = " ".join(answer.casefold().split())
    return REFUSAL.casefold() in normalized


def evaluate(system, k=3):
    """Генерирует и оценивает ответы на 20 запросов."""
    rows = []
    for item in queries:
        query = item["query"]
        answer, ids = rag_answer(query, system, k=k)
        context = build_context(ids)
        faith, faith_reason = judge_faithfulness(context, answer)
        rel, rel_reason = judge_relevance(query, answer)
        relevant = set(item["relevant_ids"])
        found = relevant.intersection(ids)
        coverage = len(found) / len(relevant) if relevant else None
        rows.append(
            {
                "query": query,
                "answer": answer,
                "ids": ids,
                "relevant_ids": item["relevant_ids"],
                "in_base": bool(relevant),
                "coverage": coverage,
                "refusal": is_refusal(answer),
                "faith": faith,
                "rel": rel,
                "faith_reason": faith_reason,
                "rel_reason": rel_reason,
            }
        )
    return rows


def diagnose(row):
    """Связывает наблюдаемую проблему с компонентом пайплайна."""
    if not row["in_base"]:
        if row["refusal"]:
            return "пробел в данных, отказ уместен"
        return "пробел в данных + ошибка генерации"

    if row["coverage"] < 1:
        if row["faith"] < 4:
            return "неполный retrieval + галлюцинация генератора"
        if row["rel"] < 3:
            return "неполный retrieval + нерелевантный ответ"
        return "неполный retrieval; метрики генерации не заметили ошибку"

    if row["faith"] < 4:
        return "ошибка генерации: галлюцинация при полном контексте"
    if row["rel"] < 3:
        return "ошибка генерации: нерелевантный ответ или ложный отказ"
    return "наблюдаемой проблемы нет"

def needs_review(row):
    """Отбирает ответы для ручного разбора."""
    retrieval_failed = row["in_base"] and row["coverage"] < 1
    return (
        row["faith"] < 4
        or row["rel"] < 3
        or retrieval_failed
        or not row["in_base"]
    )


def show_case(row):
    """Печатает доказательства и диагноз одного ответа."""
    coverage = row["coverage"]
    coverage_text = "нет эталона" if coverage is None else f"{coverage:.0%}"
    print("\nВопрос:", row["query"])
    print("Ответ:", row["answer"])
    print("Найденные id:", row["ids"])
    print("Нужные id:", row["relevant_ids"])
    print("Покрытие:", coverage_text)
    print("Faithfulness:", row["faith"], "—", row["faith_reason"])
    print("Answer relevance:", row["rel"], "—", row["rel_reason"])
    print("Диагноз:", diagnose(row))


baseline = evaluate(HANDED_OVER, k=3)
for row in baseline:
    if needs_review(row):
        show_case(row)

Сданный пайплайн {'mean_faith': 3.65, 'faith_lt4': 10, 'actionable_rel_lt3': 3, 'hit': 14, 'full_coverage': 12, 'proper_refusals': 0, 'false_refusals': 0}
Заземление, k=3 {'mean_faith': 5.0, 'faith_lt4': 0, 'actionable_rel_lt3': 3, 'hit': 14, 'full_coverage': 12, 'proper_refusals': 4, 'false_refusals': 0}
Заземление, k=8 {'mean_faith': 5.0, 'faith_lt4': 0, 'actionable_rel_lt3': 1, 'hit': 16, 'full_coverage': 15, 'proper_refusals': 4, 'false_refusals': 0}


In [ ]:
GROUNDED = (
    "Ты — ассистент поддержки. Сначала проверь, содержат ли фрагменты "
    "ответ. Если содержат, ответь только по ним и не отказывайся. "
    "Если ответа нет, ответь ровно: «В базе знаний ответа нет». "
    "Ничего не добавляй."
)


def summarize(rows):
    """Возвращает сопоставимые показатели одного прогона."""
    # Покрытие retrieval осмысленно только там, где ответ в базе есть:
    # у остальных вопросов coverage равен None
    with_answer = [row for row in rows if row["in_base"]]
    # Уместный отказ: ответа в базе нет, и система это честно сказала.
    # Это не поломка, а правильное поведение
    proper_refusal = [
        row for row in rows
        if not row["in_base"] and row["refusal"]
    ]
    # Низкая relevance, требующая действий: тот же уместный отказ
    # тоже получает низкую оценку, но чинить в нём нечего — исключаем его
    actionable_low_rel = [
        row for row in rows
        if row["rel"] < 3
        and not (not row["in_base"] and row["refusal"])
    ]
    return {
        # Средний балл — только для сравнения версий между собой,
        # не как самостоятельный показатель качества
        "mean_faith": round(
            float(np.mean([row["faith"] for row in rows])),
            2,
        ),
        # Главный показатель галлюцинаций: сколько ответов ниже порога 4
        "faith_lt4": sum(row["faith"] < 4 for row in rows),
        "actionable_rel_lt3": len(actionable_low_rel),
        # Поиск принёс хотя бы один нужный чанк
        "hit": sum(row["coverage"] > 0 for row in with_answer),
        # Поиск принёс все нужные чанки
        "full_coverage": sum(
            row["coverage"] == 1 for row in with_answer
        ),
        "proper_refusals": len(proper_refusal),
        # Ложный отказ: нужные чанки в контексте были все,
        # а система всё равно ответила «в базе знаний ответа нет».
        # Это регрессия, которую приносит заземляющий промпт
        "false_refusals": sum(
            row["in_base"]
            and row["coverage"] == 1
            and row["refusal"]
            for row in rows
        ),
    }


after_prompt = evaluate(GROUNDED, k=3)
after_prompt_and_k = evaluate(GROUNDED, k=8)

for name, rows in (
    ("Сданный пайплайн", baseline),
    ("Заземление, k=3", after_prompt),
    ("Заземление, k=8", after_prompt_and_k),
):
    print(name, summarize(rows))